# SiO₂ 涂层变角光谱椭偏（VASE）— TMM 算例

复现 LightTrans VirtualLab 用例 [VASE Analysis of a SiO₂-Coating](https://www.lighttrans.com/fileadmin/shared/UseCases/Application_UC_VASE%20Analysis%20of%20a%20SiO2-Coating.pdf)（MISC.0096），参数遵循 Woollam et al. Proc. SPIE 10294, 1029402 (1999)。

**膜系**（入射侧 → 基底）：`air(∞) | SiO₂(10 nm) | Si(∞)`（VirtualLab 物理 baseline）。

**椭偏**：ρ = r_p/r_s = tan(Ψ)·exp(iΔ)。内部计算 Δ_internal = arg(−ρ)；**绘图对齐 VirtualLab** 用 Δ_VL = 180° − Δ_internal（见 `00_software_alignment_skills.md`）。

**材料 nk（baseline）**：SiO₂ Woollam Cauchy；Si Aspnes（`simulation_database` materials / refractive_index_info）。

**运行前提**：在 `simulation_core` 根目录执行 `source scripts/init-simulation-build-env.sh build`，再 `./assets/ipynb/simulation/TMM/run_tmm.sh jupyter`（或在已 source 的环境中打开本 notebook）。须具备 `SIMULATION_ARTIFACTS_DIR`（Release `build/`）与 `SIMULATION_DATABASE_DIR`（YAML `assets/database`）；**勿**使用 `init-toykits-build-env.sh` / `.simulation_toolkits`。


In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np

from oghma_runtime import bootstrap_tmm_session

_, RUNTIME, TMM_DIR = bootstrap_tmm_session()
import simulation

from coating_optimization_utils import get_r_t, layers_from_x, stack_from_formula
from oghma_core import compare_metrics
from coating_visualizer import build_tmm_layers, layers_from_formula, plot_coating_stack
from coating_solver_test_util import make_iso_material
from simulation_database_parser import materials_db_from_token_paths


In [ ]:
# Woollam Cauchy (SPIE 10294), lambda in um
CAUCHY_A = 1.44
CAUCHY_B = 4.22e-3
CAUCHY_C = 1.89e-5


def sio2_nk_cauchy_woollam(wl_um: float) -> complex:
    wl2 = float(wl_um) ** 2
    n = CAUCHY_A + CAUCHY_B / wl2 + CAUCHY_C / (wl2 * wl2)
    return complex(n, 0.0)


MATERIALS_DB = materials_db_from_token_paths({
    "Si": [
        "rii",
        "materials",
        "main",
        "Si",
        "Si_Aspnes.yml",
    ],
})
for _name, _nk in (
    ("Air", 1.0 + 0.0j),
    ("Air_pad", 1.0 + 0.0j),
    ("SiO2_top", 1.46 + 0.0j),
    ("SiO2", 1.46 + 0.0j),
):
    MATERIALS_DB[_name] = make_iso_material(simulation, _nk, _name)

DEFAULT_FORMULA = "Air 0 SiO2 {t} Si 0"

BOUNDARY_CASES = {
    "air__si": {
        "formula": "Air 0 SiO2 {t} Si 0",
        "incident_label": "air(inf)",
        "exit_label": "Si(inf)",
        "top": "air",
        "bottom": "si",
    },
    "air__air": {
        "formula": "Air 0 SiO2 {t} Si 0 Air_pad 0",
        "incident_label": "air(inf)",
        "exit_label": "air(inf)",
        "top": "air",
        "bottom": "air",
    },
    "sio2__si": {
        "formula": "SiO2_top 0 SiO2 {t} Si 0",
        "incident_label": "SiO2(inf)",
        "exit_label": "Si(inf)",
        "top": "sio2",
        "bottom": "si",
    },
    "sio2__air": {
        "formula": "SiO2_top 0 SiO2 {t} Si 0 Air_pad 0",
        "incident_label": "SiO2(inf)",
        "exit_label": "air(inf)",
        "top": "sio2",
        "bottom": "air",
    },
}

SIO2_THICKNESS_NM = 10.0
SIO2_THICKNESS_SENS_NM = 10.1
SIO2_THICKNESS_UM = SIO2_THICKNESS_NM / 1000.0
SIO2_THICKNESS_SENS_UM = SIO2_THICKNESS_SENS_NM / 1000.0

ANGLES_DEG = [65,67.5,70,72.5,75]

PRIMARY_ANGLE_DEG = 75.0

WL_NM = np.linspace(200.0, 1000.0, 161)
WL_UM = WL_NM / 1000.0

PSI_RESOLUTION_DEG = 0.02
DELTA_RESOLUTION_DEG = 0.1
DELTA_BASELINE_OFFSET_DEG = 180
BOUNDARY_RMSE_FALLBACK = 1e-3
PRIMARY_BOUNDARY = None

from simulation_database_parser import get_simulation_database

get_simulation_database(init=True)
print("simulation database initialized")
print("SiO2 thickness (nm):", SIO2_THICKNESS_NM)
print("VASE angles (deg):", list(ANGLES_DEG))


In [3]:
def set_layer_nk(layer, nk: complex, name: str):
    layer.background_material = make_iso_material(simulation, complex(nk), name)


def delta_for_baseline_plot(delta_internal_deg):
    """Map internal Delta=arg(-rho) to VirtualLab baseline plot (180 deg - Delta_internal)."""
    delta_deg = DELTA_BASELINE_OFFSET_DEG - np.asarray(delta_internal_deg, dtype=float)
    delta_deg = np.where(delta_deg < 0, delta_deg + 360.0, delta_deg)
    return delta_deg


def _apply_layer_nk(layers, wl_um: float):
    wl_um = float(wl_um)
    sio2_nk = sio2_nk_cauchy_woollam(wl_um)
    for layer in layers:
        name = layer.background_material.name
        if name in ("SiO2", "SiO2_top"):
            set_layer_nk(layer, sio2_nk, name)
        elif name in ("Air", "Air_pad"):
            set_layer_nk(layer, 1.0 + 0.0j, name)


def build_layers_for_boundary(
    boundary_key: str | None,
    SIO2_THICKNESS_UM: float,
    wl_um: float,
):
    if boundary_key is None:
        formula = DEFAULT_FORMULA.format(t=f"{SIO2_THICKNESS_UM:.5f}")
    else:
        formula = BOUNDARY_CASES[boundary_key]["formula"].format(t=f"{SIO2_THICKNESS_UM:.5f}")
    spec = stack_from_formula(formula, MATERIALS_DB)
    sio2_idx = next(i for i, m in enumerate(spec.materials) if m.name == "SiO2")
    film_idx = spec.film_indices.index(sio2_idx)
    x = np.zeros(len(spec.film_indices), dtype=float)
    x[film_idx] = float(SIO2_THICKNESS_UM)
    layers = layers_from_x(spec, x)
    _apply_layer_nk(layers, wl_um)
    return layers, spec


def build_layers(SIO2_THICKNESS_UM: float, wl_um: float, boundary_key=None):
    if boundary_key is None:
        boundary_key = PRIMARY_BOUNDARY
    return build_layers_for_boundary(boundary_key, SIO2_THICKNESS_UM, wl_um)[0]


def compute_psi_delta(layers, angle_deg: float, wl_um: float):
    r_s, _, r_p, _ = get_r_t(layers, np.deg2rad(angle_deg), float(wl_um))
    rho = r_p / r_s
    psi = np.rad2deg(np.arctan(abs(rho)))
    delta = np.rad2deg(np.angle(-rho))
    delta_vl = np.rad2deg(np.angle(rho))
    return psi, delta, delta_vl, rho, r_s, r_p


def ellipsometry_spectrum(SIO2_THICKNESS_UM: float, angle_deg: float, boundary_key=None):
    psi = np.zeros(len(WL_UM), dtype=float)
    delta = np.zeros(len(WL_UM), dtype=float)
    for i, wl in enumerate(WL_UM):
        layers = build_layers(SIO2_THICKNESS_UM, float(wl), boundary_key=boundary_key)
        psi[i], delta[i], _, _, _, _ = compute_psi_delta(layers, angle_deg, float(wl))
    return psi, delta


def vase_spectra(SIO2_THICKNESS_UM: float, angles_deg: np.ndarray, boundary_key=None):
    psi_map, delta_map = {}, {}
    for ang in angles_deg:
        psi_map[float(ang)], delta_map[float(ang)] = ellipsometry_spectrum(
            SIO2_THICKNESS_UM, float(ang), boundary_key=boundary_key
        )
    return psi_map, delta_map


def boundary_score(boundary_key: str, ref_psi, ref_delta_vl):
    psi, delta = ellipsometry_spectrum(SIO2_THICKNESS_UM, PRIMARY_ANGLE_DEG, boundary_key=boundary_key)
    delta_vl = delta_for_baseline_plot(delta)
    s_psi = compare_metrics(psi, ref_psi, x=WL_NM)
    s_delta = compare_metrics(delta_vl, ref_delta_vl, x=WL_NM)
    return float(s_psi["rmse"] + s_delta["rmse"]), s_psi, s_delta


def select_primary_boundary():
    ref_psi, ref_delta = ellipsometry_spectrum(
        SIO2_THICKNESS_UM, PRIMARY_ANGLE_DEG, boundary_key=None
    )
    ref_delta_vl = delta_for_baseline_plot(ref_delta)
    scores = {}
    details = {}
    for key in BOUNDARY_CASES:
        score, s_psi, s_delta = boundary_score(key, ref_psi, ref_delta_vl)
        scores[key] = score
        details[key] = (s_psi, s_delta)
    best_key = min(scores, key=scores.get)
    if scores[best_key] > BOUNDARY_RMSE_FALLBACK:
        return None, scores, details, ref_psi, ref_delta_vl
    return best_key, scores, details, ref_psi, ref_delta_vl


def plot_boundary_stack_on_ax(ax, boundary_key: str, *, title: str, halfspace_margin_nm: float = 40.0):
    case = BOUNDARY_CASES[boundary_key]
    thicknesses_nm = [0.0, SIO2_THICKNESS_NM, 0.0]
    labels = ["SiO2", "SiO2"]
    z = np.cumsum([0.0] + thicknesses_nm[1:-1])
    colors = plt.cm.tab20(np.linspace(0, 1, len(labels)))
    for k, (z0, z1, lab) in enumerate(zip(z, z[1:], labels)):
        if z1 <= z0:
            continue
        ax.barh(0, z1 - z0, left=z0, height=0.5, color=colors[k], edgecolor="k")
        ax.text(0.5 * (z0 + z1), 0, lab, ha="center", va="center", fontsize=7)
    ax.annotate("", xy=(0, 0.35), xytext=(-halfspace_margin_nm * 0.75, 0.35),
                arrowprops=dict(arrowstyle="<-", color="gray"))
    ax.text(-halfspace_margin_nm * 0.375, 0.55, case["incident_label"],
            ha="center", fontsize=7, color="gray")
    ax.annotate("", xy=(z[-1], 0.35), xytext=(z[-1] + halfspace_margin_nm * 0.75, 0.35),
                arrowprops=dict(arrowstyle="->", color="gray"))
    ax.text(z[-1] + halfspace_margin_nm * 0.375, 0.55, case["exit_label"],
            ha="center", fontsize=7, color="gray")
    ax.set_xlim(-halfspace_margin_nm, z[-1] + halfspace_margin_nm)
    ax.set_yticks([])
    ax.set_xlabel("TMM depth z (nm)")
    ax.set_title(title)


def plot_vase_boundary_stack(boundary_key: str, *, title: str):
    case = BOUNDARY_CASES[boundary_key]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        formula = case["formula"].format(t=f"{SIO2_THICKNESS_UM:.5f}")
        mats, th = layers_from_formula(formula, MATERIALS_DB, simulation_module=simulation)
        plot_coating_stack(build_tmm_layers(mats, th, simulation_module=simulation), title=title)


def plot_1d_compare(x, tmm, baseline, ylabel, title, baseline_name="reference"):
    err = tmm - baseline
    fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    axes[0].plot(x, tmm, label="TMM")
    axes[0].plot(x, baseline, "--", label=baseline_name)
    axes[0].set_ylabel(ylabel)
    axes[0].set_title(title)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(x, err, color="C3")
    axes[1].axhline(0, color="k", lw=0.5)
    axes[1].set_xlabel("Wavelength (nm)")
    axes[1].set_ylabel(f"err {ylabel}")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return err


## 膜系结构


In [ ]:

formula = DEFAULT_FORMULA.format(t=f"{SIO2_THICKNESS_UM:.5f}")
mats, th = layers_from_formula(formula, MATERIALS_DB, simulation_module=simulation)
plot_coating_stack(
    build_tmm_layers(mats, th, simulation_module=simulation),
    title="VASE stack (default): air / SiO2(10nm) / Si",
)


## 材料

SiO₂：内联 Woollam Cauchy；Si：`nk_at_wavelength` + `simulation_database` materials（Aspnes）。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

sio2_n = [sio2_nk_cauchy_woollam(w).real for w in WL_UM]
si_mat = MATERIALS_DB["Si"]
si_n = [complex(si_mat.nk_at_wavelength_um(w)).real for w in WL_UM]
si_k = [complex(si_mat.nk_at_wavelength_um(w)).imag for w in WL_UM]

axes[0].plot(WL_NM, sio2_n, "C0", lw=1.8, label="SiO2 Cauchy (Woollam)")
axes[0].plot(WL_NM, si_n, "C1", lw=1.2, label="Si Aspnes (query path)")
axes[1].plot(WL_NM, si_k, "C1", lw=1.2, label="Si Aspnes (query path)")
axes[0].axhline(0.0, color="C3", ls="--", lw=0.8, alpha=0.5, label="SiO2 k=0")

axes[0].set_ylabel("n")
axes[1].set_ylabel("k")
axes[1].set_xlabel("Wavelength (nm)")
axes[0].set_title("Layer refractive indices (baseline materials)")
axes[0].legend(fontsize=8)
axes[1].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## VASE 椭偏系数

10 nm SiO₂/Si，65°–75°：Ψ 与 Δ（**Δ 为 VirtualLab 约定，deg**）。


In [ ]:
psi_vase, delta_vase = vase_spectra(SIO2_THICKNESS_UM, ANGLES_DEG)

fig, axes = plt.subplots(1, 2)
for ang in ANGLES_DEG:
    label = f"{ang:.0f} deg"
    axes[0].plot(WL_NM, psi_vase[float(ang)], lw=1.5, label=label)
    axes[1].plot(WL_NM, delta_for_baseline_plot(delta_vase[float(ang)]), lw=1.5, label=label)

axes[0].set_ylabel(r"$\Psi$")
axes[1].set_ylabel(r"$\Delta$")
axes[1].set_xlabel("Wavelength (nm)")
axes[0].set_title(
    f"Ellipsometry: air/SiO2({SIO2_THICKNESS_NM:.0f} nm)/Si"
)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].grid(True, alpha=0.3)
plt.show()


In [ ]:

from IPython.display import Image, display
from pathlib import Path
_ref = Path('../resource/tmm/virtuallab_ellipsometry.png')
if _ref.is_file():
    display(Image(filename=str(_ref)))
else:
    print(f'reference image not found: {_ref}')

## 厚度灵敏度（1 Å 变化）

对比 10.0 nm 与 10.1 nm；Woollam 分辨率：Ψ ≈ 0.02°，Δ ≈ 0.1°。


In [ ]:
psi_10, delta_10 = ellipsometry_spectrum(SIO2_THICKNESS_UM, PRIMARY_ANGLE_DEG)
psi_101, delta_101 = ellipsometry_spectrum(SIO2_THICKNESS_SENS_UM, PRIMARY_ANGLE_DEG)
delta_10_vl = delta_for_baseline_plot(delta_10)
delta_101_vl = delta_for_baseline_plot(delta_101)
dpsi = psi_101 - psi_10
ddelta = delta_101_vl - delta_10_vl

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True)

panels = [
    (psi_10, delta_10_vl, f"{SIO2_THICKNESS_NM:.1f} nm", False, False),
    (psi_101, delta_101_vl, f"{SIO2_THICKNESS_SENS_NM:.1f} nm", False, False),
    (dpsi, ddelta, r"$\Delta\Psi$ / $\Delta\Delta$", True, True),
]

for i, (psi_y, delta_y, title, show_psi_res, show_delta_res) in enumerate(panels):
    ax = axes[i]
    ax.plot(WL_NM, psi_y, "C0", lw=1.5)
    if i == 0:
        ax.set_ylabel(r"$\Psi$ (deg)", color="C0")
    ax.tick_params(axis="y", labelcolor="C0")
    ax.set_title(title)
    ax.set_xlabel("Wavelength (nm)")
    ax.grid(True, alpha=0.3)
    if show_psi_res:
        ax.axhline(PSI_RESOLUTION_DEG, color="k", ls="--", lw=1)
        ax.axhline(-PSI_RESOLUTION_DEG, color="k", ls="--", lw=1)

    ax2 = ax.twinx()
    ax2.plot(WL_NM, delta_y, "C3", lw=1.5)
    if i == 2:
        ax2.set_ylabel(r"$\Delta$ (deg, VL)", color="C3")
    ax2.tick_params(axis="y", labelcolor="C3")
    if show_delta_res:
        ax2.axhline(DELTA_RESOLUTION_DEG, color="k", ls="--", lw=1)
        ax2.axhline(-DELTA_RESOLUTION_DEG, color="k", ls="--", lw=1)

fig.suptitle(
    f"Thickness sensitivity @ {PRIMARY_ANGLE_DEG:.0f} deg: "
    f"{SIO2_THICKNESS_NM:.1f} nm vs {SIO2_THICKNESS_SENS_NM:.1f} nm (1 A)",
    y=1.02,
)
fig.tight_layout()
plt.show()


In [ ]:
from IPython.display import Image, display
from pathlib import Path
_ref = Path('../resource/tmm/virtuallab_ellipsometry_sensitivity.png')
if _ref.is_file():
    display(Image(filename=str(_ref)))
else:
    print(f'reference image not found: {_ref}')
